In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch12-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "12-02-experimental-design.py"

# 八個反應名額，五種初期設計

比較溫度與反應時間的候選安排。這裡只排實驗，不產生產率。修改分析格後，請接著執行最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
from itertools import product
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import qmc

# 規劃八組條件，不建立或模擬產率欄。
seed = 42
n = 8
ranges = np.array([[40., 80.], [20., 60.]])  # 溫度 °C、反應時間 min
plans = {
    "Factorial (4 x 2)": np.array(list(product(np.linspace(0, 1, 4), [0., 1.]))),
    "Random": np.random.default_rng(seed).random((n, 2)),
    "Latin hypercube": qmc.LatinHypercube(d=2, seed=seed).random(n),
    "Sobol": qmc.Sobol(d=2, scramble=True, seed=seed).random_base2(m=3),
    "Halton": qmc.Halton(d=2, scramble=True, seed=seed).random(n),
}

def physical(points):
    return ranges[:, 0] + points * (ranges[:, 1] - ranges[:, 0])

def covered(points):
    bins = np.minimum((points * n).astype(int), n - 1)
    return tuple(len(np.unique(axis)) for axis in bins.T)

fig, panels = plt.subplots(2, 3, figsize=(10, 5.8), constrained_layout=True,
                           sharex=True, sharey=True)
for ax, (name, points) in zip(panels.flat, plans.items()):
    temperature, time = physical(points).T
    assert len(points) == n and np.all((points >= 0) & (points <= 1))
    print(name, "occupied temperature/time bands:", covered(points))
    ax.scatter(temperature, time, color="#136b71", s=38)
    ax.set(title=name, xlim=(37, 83), ylim=(17, 63))
    ax.set_xticks([40, 50, 60, 70, 80])
    ax.set_yticks([20, 30, 40, 50, 60])
    ax.grid(alpha=.22)
panels.flat[-1].axis("off")
fig.supxlabel("Temperature (°C)")
fig.supylabel("Reaction time (min)")
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out / "12-02-experimental-design.png", dpi=160)


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.removeprefix("This module requires PyTorch to be installed.\n").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "12-02-experimental-design.png")))